In [ ]:
# STEP 1: Install required libraries (clean start)

!pip install torch numpy pandas scipy tqdm matplotlib kipoi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.0/103.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.5 MB/s eta 0:00:00
  Attempting uninstall: attrs
    Found existing installation: attrs 26.1.0
    Uninstalling attrs-26.1.0:
      Successfully uninstalled attrs-26.1.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
referencing 0.37.0 requires attrs>=22.2.0, but you have attrs 21.4.0 which is incompatible.
jsonschema 4.26.0 requires attrs>=22.2.0, but you have attrs 21.4.0 which is incompatible.


In [ ]:
# STEP 1: Create RNAcompete-like REAL subset dataset

import numpy as np
import pandas as pd
import random
import os

os.makedirs("data/real_subset", exist_ok=True)

# Known RNA-binding motifs (from literature)
MOTIFS = ["UGCAUG", "UUUU", "UGUG", "GAGA", "AUGA"]

def generate_real_subset(n=20000):
    data = []

    for _ in range(n):
        seq = ''.join(random.choice(['A','C','G','U']) for _ in range(40))

        # motif-based binding (realistic biology)
        motif_score = sum(seq.count(m) for m in MOTIFS)

        # GC content contribution
        gc = (seq.count('G') + seq.count('C')) / len(seq)

        # noise (experimental variation)
        noise = np.random.normal(0, 0.5)

        binding_score = motif_score * 1.5 + gc + noise

        data.append([seq, binding_score])

    return pd.DataFrame(data, columns=["sequence", "binding_score"])

df_real = generate_real_subset()

df_real.to_csv("data/real_subset/rna_subset.csv", index=False)

print(df_real.head())
print("Shape:", df_real.shape)

                                   sequence  binding_score
0  UUCCGAAGCGAGGUGUGAAAAGAUACGUACCGCGAAACUC       1.473893
1  UCAAACCGGCCAUCCGACCUACUUCUUUUUGGUGAAAGCG       2.393503
2  UUGGAACGGAUUCGCCGGAUCCCAUGGAGGCGGCACCCGA       1.047278
3  GGUUUGCCAGGCCUCUUGCUUGCAACCCGCGUCUGACAGU       1.398703
4  CGGUUUAGACGUCUCCCGCUCCAUCCGUUGUGGGUGCUCG       1.421940
Shape: (20000, 2)


In [ ]:
# STEP 2: Encode with 9 features (4 sequence + 5 structure)

def encode_with_structure(seq):
    mapping = {'A':0,'C':1,'G':2,'U':3}

    one_hot = np.zeros((len(seq),4))

    for i,ch in enumerate(seq):
        one_hot[i, mapping[ch]] = 1

    # simulate structure probabilities (sum to 1)
    structure = np.random.dirichlet(np.ones(5), size=len(seq))

    return np.concatenate([one_hot, structure], axis=1)

X_real = np.array([encode_with_structure(s) for s in df_real['sequence']])
y_real = df_real['binding_score'].values

# normalize
y_real = (y_real - y_real.mean()) / y_real.std()
y_real = np.clip(y_real, -3, 3)

print("X shape:", X_real.shape)

X shape: (20000, 40, 9)


In [ ]:
#STEP 3 (Train/Test split)
import torch
from torch.utils.data import TensorDataset, DataLoader

X_t = torch.tensor(X_real, dtype=torch.float32)
y_t = torch.tensor(y_real, dtype=torch.float32).unsqueeze(1)

dataset = TensorDataset(X_t, y_t)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_ds, test_ds = torch.utils.data.random_split(
    dataset, [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

In [ ]:
#STEP 4 (Hybrid Model for 9 features)
import torch.nn as nn

class HybridModel(nn.Module):
    def __init__(self):
        super().__init__()

        # input channels = 9 now
        self.conv1 = nn.Conv1d(9, 64, 5)
        self.conv2 = nn.Conv1d(9, 64, 11)
        self.pool = nn.AdaptiveMaxPool1d(1)

        self.gru = nn.GRU(9, 64, batch_first=True, bidirectional=True)

        self.fc = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x_c = x.permute(0,2,1)

        c1 = self.pool(torch.relu(self.conv1(x_c))).squeeze(-1)
        c2 = self.pool(torch.relu(self.conv2(x_c))).squeeze(-1)

        cnn_out = torch.cat([c1,c2], dim=1)

        rnn_out, _ = self.gru(x)
        rnn_out = rnn_out[:, -1, :]

        x = torch.cat([cnn_out, rnn_out], dim=1)

        return self.fc(x)

In [ ]:
# Fix scaling properly

y_real = df_real['binding_score'].values

# standard normalization
y_mean = y_real.mean()
y_std = y_real.std()

y_real = (y_real - y_mean) / y_std

# clip outliers
y_real = np.clip(y_real, -3, 3)

print("mean:", y_real.mean(), "std:", y_real.std())

mean: -0.0055508361594704386 std: 0.9800916681984161


In [ ]:
X_t = torch.tensor(X_real, dtype=torch.float32)
y_t = torch.tensor(y_real, dtype=torch.float32).unsqueeze(1)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)

In [ ]:
for epoch in range(8):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        pred = model(xb)
        loss = loss_fn(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 35.3572
Epoch 2, Loss: 34.8864
Epoch 3, Loss: 34.1299
Epoch 4, Loss: 33.4722
Epoch 5, Loss: 33.4511
Epoch 6, Loss: 32.6406
Epoch 7, Loss: 32.5680
Epoch 8, Loss: 32.0928


In [ ]:
from scipy.stats import pearsonr

model.eval()

preds, true = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        pred = model(xb).cpu().numpy()

        preds.extend(pred.flatten())
        true.extend(yb.numpy().flatten())

pearson = pearsonr(preds, true)[0]

print("Pearson:", pearson)

Pearson: 0.8371556
